# EXP-010 | 앙상블 (LightGBM + CatBoost + XGBoost)

EXP-004(LGB Optuna), EXP-006(CatBoost), EXP-007(XGBoost) 세 모델을 앙상블.

| 앙상블 방법 | 설명 |
|---|---|
| Simple Average | 동일 가중치 평균 |
| AUC-weighted Average | OOF AUC 기반 가중 평균 |
| Rank Average | 확률 → 순위 변환 후 평균 (분포 차이 보정) |
| Optuna Weight Search | 최적 가중치 탐색 |

| 항목 | 내용 |
|---|---|
| 기반 실험 | EXP-004 / EXP-006 / EXP-007 |
| CV | Stratified 5-Fold |

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, accuracy_score, average_precision_score
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
EXP_NO   = 10
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 85)  /  X_test: (90067, 85)


## 2. 개별 모델 학습 (OOF + Test 예측 저장)

In [3]:
# ── LightGBM (EXP-004 파라미터) ──────────────────────────
LGB_PARAMS = dict(
    objective='binary', metric='auc', verbosity=-1, seed=SEED,
    is_unbalance=True, learning_rate=0.05, num_leaves=127,
    min_child_samples=50, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, lambda_l1=0.1, lambda_l2=0.1,
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_lgb  = np.zeros(len(X_train))
test_lgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    ds_tr  = lgb.Dataset(X_tr, label=y_tr)
    ds_val = lgb.Dataset(X_val, label=y_val, reference=ds_tr)
    model  = lgb.train(LGB_PARAMS, ds_tr, num_boost_round=2000,
                       valid_sets=[ds_val],
                       callbacks=[lgb.early_stopping(100, verbose=False),
                                  lgb.log_evaluation(period=-1)])
    oof_lgb[val_idx]  = model.predict(X_val)
    test_lgb         += model.predict(X_test) / N_FOLDS

auc_lgb = roc_auc_score(y_train, oof_lgb)
print(f'LightGBM  OOF AUC: {auc_lgb:.5f}')

LightGBM  OOF AUC: 0.73864


In [4]:
# ── CatBoost (EXP-006 파라미터) ──────────────────────────
oof_cat  = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = CatBoostClassifier(
        iterations=2000, learning_rate=0.05, depth=6, l2_leaf_reg=3,
        loss_function='Logloss', eval_metric='AUC',
        auto_class_weights='Balanced', random_seed=SEED,
        verbose=False, early_stopping_rounds=100,
    )
    model.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    oof_cat[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_cat         += model.predict_proba(X_test)[:, 1] / N_FOLDS

auc_cat = roc_auc_score(y_train, oof_cat)
print(f'CatBoost  OOF AUC: {auc_cat:.5f}')

CatBoost  OOF AUC: 0.74003


In [5]:
# ── XGBoost (EXP-007 파라미터) ───────────────────────────
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
oof_xgb  = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(
        n_estimators=2000, learning_rate=0.05, max_depth=6,
        min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
        reg_alpha=0.1, reg_lambda=0.1, scale_pos_weight=neg_pos_ratio,
        eval_metric='auc', tree_method='hist', random_state=SEED,
        verbosity=0, early_stopping_rounds=100,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1]
    test_xgb         += model.predict_proba(X_test)[:, 1] / N_FOLDS

auc_xgb = roc_auc_score(y_train, oof_xgb)
print(f'XGBoost   OOF AUC: {auc_xgb:.5f}')

print(f'\n개별 모델 요약')
print(f'  LightGBM : {auc_lgb:.5f}')
print(f'  CatBoost : {auc_cat:.5f}')
print(f'  XGBoost  : {auc_xgb:.5f}')

XGBoost   OOF AUC: 0.74016

개별 모델 요약
  LightGBM : 0.73864
  CatBoost : 0.74003
  XGBoost  : 0.74016


## 3. 앙상블 방법 비교

In [6]:
oofs  = np.stack([oof_lgb,  oof_cat,  oof_xgb],  axis=1)  # (n, 3)
tests = np.stack([test_lgb, test_cat, test_xgb], axis=1)  # (n, 3)
aucs  = np.array([auc_lgb, auc_cat, auc_xgb])

results = {}

# ── 1. Simple Average ─────────────────────────────────────
oof_simple = oofs.mean(axis=1)
results['Simple Average'] = (roc_auc_score(y_train, oof_simple),
                              tests.mean(axis=1))

# ── 2. AUC-weighted Average ───────────────────────────────
w_auc = aucs / aucs.sum()
oof_weighted = (oofs * w_auc).sum(axis=1)
results['AUC-weighted'] = (roc_auc_score(y_train, oof_weighted),
                            (tests * w_auc).sum(axis=1))

# ── 3. Rank Average ───────────────────────────────────────
from scipy.stats import rankdata
oof_ranks  = np.stack([rankdata(oofs[:, i])  for i in range(3)], axis=1)
test_ranks = np.stack([rankdata(tests[:, i]) for i in range(3)], axis=1)
oof_rank_avg  = oof_ranks.mean(axis=1)
results['Rank Average'] = (roc_auc_score(y_train, oof_rank_avg),
                            test_ranks.mean(axis=1))

# ── 4. Optuna 최적 가중치 탐색 ───────────────────────────
def optuna_objective(trial):
    w = np.array([
        trial.suggest_float('w_lgb', 0.0, 1.0),
        trial.suggest_float('w_cat', 0.0, 1.0),
        trial.suggest_float('w_xgb', 0.0, 1.0),
    ])
    w = w / w.sum()
    return roc_auc_score(y_train, (oofs * w).sum(axis=1))

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(optuna_objective, n_trials=300, show_progress_bar=False)

best_w = np.array([study.best_params['w_lgb'],
                   study.best_params['w_cat'],
                   study.best_params['w_xgb']])
best_w = best_w / best_w.sum()
oof_optuna = (oofs * best_w).sum(axis=1)
results['Optuna Weights'] = (roc_auc_score(y_train, oof_optuna),
                              (tests * best_w).sum(axis=1))

# ── 결과 출력 ─────────────────────────────────────────────
print('=' * 55)
print(f'  개별 모델     LGB={auc_lgb:.5f}  CAT={auc_cat:.5f}  XGB={auc_xgb:.5f}')
print('-' * 55)
best_method, best_auc, best_test = '', 0, None
for method, (auc, test_pred) in results.items():
    flag = ' ←'if auc > max(aucs) else ''
    print(f'  {method:20s}: {auc:.5f}{flag}')
    if auc > best_auc:
        best_auc, best_method, best_test = auc, method, test_pred
print('=' * 55)
print(f'\n최적 방법: {best_method}  AUC={best_auc:.5f}')
print(f'Optuna 최적 가중치  LGB={best_w[0]:.3f}  CAT={best_w[1]:.3f}  XGB={best_w[2]:.3f}')

  개별 모델     LGB=0.73864  CAT=0.74003  XGB=0.74016
-------------------------------------------------------
  Simple Average      : 0.74031 ←
  AUC-weighted        : 0.74031 ←
  Rank Average        : 0.74033 ←
  Optuna Weights      : 0.74046 ←

최적 방법: Optuna Weights  AUC=0.74046
Optuna 최적 가중치  LGB=0.063  CAT=0.419  XGB=0.518


## 4. Submission 저장 & 실험 기록

In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
sub['probability'] = best_test if best_test is not None else tests.mean(axis=1)
auc_str   = f'{best_auc:.4f}'.replace('.', '')
out_fname = f'submission_exp{EXP_NO:03d}_{AUTHOR}_{auc_str}.csv'
sub.to_csv(OUT_DIR / out_fname, index=False)
print(f'저장: {OUT_DIR / out_fname}  ({best_method})')

저장: ../data/submissions/submission_exp010_yjc_07405.csv  (Optuna Weights)


In [8]:
def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, Alignment, Border, Side, PatternFill
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

oof_best   = results[best_method][0]
oof_binary = ((oofs * best_w).sum(axis=1) >= 0.5).astype(int)

params_str = (f'LGB={auc_lgb:.5f}, CAT={auc_cat:.5f}, XGB={auc_xgb:.5f} | '
              f'best={best_method} | w=({best_w[0]:.3f},{best_w[1]:.3f},{best_w[2]:.3f})')
NOTES    = f'LightGBM+CatBoost+XGBoost 앙상블. 최적 방법: {best_method}'
INSIGHTS = f'Optuna 최적 가중치 LGB={best_w[0]:.3f} CAT={best_w[1]:.3f} XGB={best_w[2]:.3f}'

log_to_leaderboard(
    EXP_NO, AUTHOR, 'Ensemble(LGB+CAT+XGB)', params_str,
    f1_score(y_train, oof_binary), recall_score(y_train, oof_binary),
    precision_score(y_train, oof_binary), accuracy_score(y_train, oof_binary),
    best_auc, CV_STR, 'v1', X_train.shape[1], 'is_unbalance / Balanced / scale_pos_weight',
    'N', None, 'notebooks/09_ensemble_yjcho.ipynb', NOTES, INSIGHTS
)

[leaderboard.xlsx] EXP-010 기록 완료 (row 10)
